## ChEBI hierarchy

This notebook extracts the hierarchy of all ChEBI entities provided by chebi.owl. The file is from 01.01.2025.
The hierarchy can reveal e.g. reveal parent-children relationships and the amount of descendandts or ancestors an entity has.

First up is the extraction of data from the file.

In [16]:
import rdflib
import pandas as pd
import networkx as nx
import pickle

In [2]:
CHEBI = f"../files/chebi.owl"
c = rdflib.Graph()
c.parse(CHEBI, format="xml")

<Graph identifier=Nbb66edc124a6431e83edef11d53cf785 (<class 'rdflib.graph.Graph'>)>

Following is the query for parent/children relations, which is needed to obtain amount of descendants an entity has.

In [8]:
query="""
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?child ?parent
WHERE {
  OPTIONAL { ?child rdfs:subClassOf ?parent }
}
"""

In [13]:
h = c.query(query)
h = pd.DataFrame(h.bindings).map(str).rename(columns=str)
h = h[h["child"].apply(lambda x: "CHEBI" in x )]
h = h[h["parent"].apply(lambda x: "CHEBI" in x )]
h = h.drop_duplicates(ignore_index=True)
h["child"] = h["child"].str.extract(r"/obo/(.+)")[0].str.replace("_", ":", regex=False)
h["parent"] = h["parent"].str.extract(r"/obo/(.+)")[0].str.replace("_", ":", regex=False)

Next up is the ontology hierarchy tree.

In [20]:
G = nx.DiGraph()
G.add_edges_from(h[["parent", "child"]].itertuples(index=False, name=None))

with open("chebi_descendants.pkl", "wb") as f:
    pickle.dump(G, f)

Here is a short example on how to apply this throughout the project. One gets all descendants for each specific submission.

In [22]:
def get_all_descendants(graph, node):
    return list(nx.descendants(graph, node))

descendants = get_all_descendants(G, "CHEBI:133004")
print(f"Total descendants: {len(descendants)}")
print(f"Descendants: {descendants}")

Total descendants: 35
Descendants: ['CHEBI:132895', 'CHEBI:9514', 'CHEBI:11', 'CHEBI:16777', 'CHEBI:38', 'CHEBI:845', 'CHEBI:5582', 'CHEBI:132893', 'CHEBI:4331', 'CHEBI:49', 'CHEBI:10', 'CHEBI:7853', 'CHEBI:80898', 'CHEBI:9774', 'CHEBI:2487', 'CHEBI:3546', 'CHEBI:36323', 'CHEBI:6642', 'CHEBI:9509', 'CHEBI:7714', 'CHEBI:5677', 'CHEBI:5568', 'CHEBI:5049', 'CHEBI:4319', 'CHEBI:9512', 'CHEBI:3346', 'CHEBI:81051', 'CHEBI:8886', 'CHEBI:8651', 'CHEBI:5996', 'CHEBI:7712', 'CHEBI:3300', 'CHEBI:2842', 'CHEBI:4323', 'CHEBI:3063']
